# 11. Conditional Harmonic Language Modeling

Notebook 9 describes how harmonic vocabulary changes. Notebook 10 embeds harmonic classes by distributional behavior. This notebook moves from description to prediction:

**Given a harmonic context and metadata such as decade, genre, or artist, what harmonic state should come next?**

The motivating question is:

> Given an `n`-gram, what would the next `n`-gram be if this song belonged to a particular artist or style?

This is the harmonic analogue of conditional NLP language modeling. In text, we estimate

$$
p(w_{t+1} \mid w_{t-n+1}, \ldots, w_t, \text{author}, \text{date}, \text{genre}).
$$

Here we estimate

$$
p(H_n(t+1) \mid H_n(t), \text{decade}, \text{genre}, \text{artist}).
$$

The first-pass model is intentionally count-based and interpretable. It builds Markov transitions between harmonic classes, then uses support-aware interpolation to back off from sparse artist evidence to genre, decade, and global evidence.

## Modeling Plan

1. Build a support-limited harmonic vocabulary from `H_n`.
2. Stream raw chord sequences and convert each song to `H_n` state sequences.
3. Count adjacent transitions: `H_n(t) -> H_n(t+1)`.
4. Count the same transitions under metadata scopes: global, decade, genre, artist.
5. Evaluate global and interpolated models on a deterministic song-level holdout.
6. Provide a style-conditioned continuation function.

This notebook predicts the next harmonic **state** rather than the next exact chord. For a chord sequence `C F G Amin`, with `n = 3`, the transition is:

```text
H_3(C F G) -> H_3(F G Amin)
```

That creates a Markov chain over harmonic classes. The result is not yet a full generative composer; it is a transparent baseline for asking whether metadata changes the continuation distribution.

## Setup

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import importlib
import math
import os
import sys

CWD = Path.cwd()
ROOT = CWD.parent if (CWD / "utils").exists() else CWD
MPLCONFIGDIR = ROOT / ".matplotlib-cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import duckdb
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = ROOT / "notebooks"
sys.path = [p for p in sys.path if p != str(NOTEBOOK_DIR)]
sys.path.insert(0, str(NOTEBOOK_DIR))
for module_name in list(sys.modules):
    if module_name == "utils" or module_name.startswith("utils."):
        del sys.modules[module_name]

from utils import duckdb_store as ds
from utils import ngram_features as nf

ds = importlib.reload(ds)
nf = importlib.reload(nf)
expected = {
    "duckdb_store": (NOTEBOOK_DIR / "utils" / "duckdb_store.py").resolve(),
    "ngram_features": (NOTEBOOK_DIR / "utils" / "ngram_features.py").resolve(),
}
loaded = {
    "duckdb_store": Path(ds.__file__).resolve(),
    "ngram_features": Path(nf.__file__).resolve(),
}
assert loaded == expected, f"Imported wrong utility module(s): {loaded}; expected {expected}"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

RAW_PATH = ROOT / "data" / "raw" / "chordonomicon_v2.csv"
DB_PATH = ROOT / "data" / "processed" / "harmonic_trends.duckdb"
OUT_DIR = ROOT / "data" / "processed" / "conditional_language_model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NS = (3, 4, 5)
MIN_GLOBAL_HARMONIC_COUNT = 1_000
TOP_HARMONIC_PER_N = 1_000
MIN_ARTIST_SONGS = 50
MAX_ARTISTS = 500
MIN_SCOPE_SOURCE_COUNT = 10
TEST_MODULUS = 5
TEST_REMAINDER = 0
MAX_TEST_EVENTS = 100_000
MAX_SONGS = None
CHUNKSIZE = 25_000
REBUILD_TRANSITION_STORE = False
TRANSITION_STORE_VERSION = "conditional_lm_v1"
TRANSITION_STORE_NAME = "full" if MAX_SONGS is None else f"sample_{MAX_SONGS}"
TRANSITION_EVENTS_TABLE = f"conditional_lm_transition_events_{TRANSITION_STORE_NAME}"
TRANSITION_METADATA_TABLE = f"conditional_lm_transition_metadata_{TRANSITION_STORE_NAME}"
TRANSITION_COUNTS_TABLE = f"conditional_lm_transition_counts_{TRANSITION_STORE_NAME}"
SOURCE_SUPPORT_TABLE = f"conditional_lm_source_support_{TRANSITION_STORE_NAME}"
SCOPE_SUMMARY_TABLE = f"conditional_lm_scope_summary_{TRANSITION_STORE_NAME}"

BACKOFF_K = 50
INTERPOLATION_PRIORS = {
    "artist_id": 0.45,
    "main_genre": 0.25,
    "decade_1950_plus": 0.15,
    "global": 0.15,
}

assert RAW_PATH.exists(), RAW_PATH
assert DB_PATH.exists(), DB_PATH

{
    "python": sys.executable,
    "duckdb_version": duckdb.__version__,
    "out_dir": str(OUT_DIR),
    **{name: str(path) for name, path in loaded.items()},
}

## What This Model Is

This is the harmonic equivalent of a smoothed `n`-gram language model.

- It is **interpretable**: every prediction comes from observed transition counts.
- It is **conditional**: decade, genre, and artist scopes can change the continuation distribution.
- It is **support-aware**: sparse artist estimates are blended with broader genre/decade/global estimates.
- It is **not yet neural**: notebook 10's embeddings are reserved for the next step, where nearby contexts can share probability mass.

The defaults prioritize a manageable first pass. Increase `TOP_HARMONIC_PER_N`, `MAX_ARTISTS`, or `NS` when you want a broader model.

## Helpers

In [ ]:
def split_for_song(song_id: int) -> str:
    return "test" if int(song_id) % TEST_MODULUS == TEST_REMAINDER else "train"


def harmonic_sequence(tokens: list[str], n: int, vocab_ids: set[str]) -> list[str]:
    sequence = []
    for ngram in nf.chord_ngrams(tokens, n):
        try:
            hid = nf.harmonic_id(nf.first_root_normalized_key(ngram))
        except Exception:
            continue
        if hid in vocab_ids:
            sequence.append(hid)
    return sequence


def song_scopes(row) -> list[tuple[str, str]]:
    scopes = [("global", "__all__")]
    if pd.notna(row.decade) and int(row.decade) >= 1950:
        scopes.append(("decade_1950_plus", str(int(row.decade))))
    if pd.notna(row.main_genre):
        scopes.append(("main_genre", str(row.main_genre)))
    if pd.notna(row.artist_id) and row.artist_id in ARTIST_SCOPE_IDS:
        scopes.append(("artist_id", str(row.artist_id)))
    return scopes


def short_label(harmonic_id: str) -> str:
    row = VOCAB_LOOKUP.get(harmonic_id)
    if row is None:
        return harmonic_id
    return f"{row['example_ngram']} ({harmonic_id.split('_', 1)[-1][:6]})"


def entropy_from_counter(counter: Counter) -> float:
    total = sum(counter.values())
    if total <= 0:
        return 0.0
    return -sum((count / total) * math.log(count / total) for count in counter.values() if count > 0)


def transition_store_signature() -> dict[str, str]:
    return {
        "version": TRANSITION_STORE_VERSION,
        "ns": ",".join(str(n) for n in NS),
        "min_global_harmonic_count": str(MIN_GLOBAL_HARMONIC_COUNT),
        "top_harmonic_per_n": str(TOP_HARMONIC_PER_N),
        "min_artist_songs": str(MIN_ARTIST_SONGS),
        "max_artists": str(MAX_ARTISTS),
        "test_modulus": str(TEST_MODULUS),
        "test_remainder": str(TEST_REMAINDER),
        "max_songs": "None" if MAX_SONGS is None else str(MAX_SONGS),
    }


def duckdb_table_exists(con: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    return bool(
        con.execute(
            """
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_name = ?
            """,
            [table_name],
        ).fetchone()[0]
    )


def transition_store_is_current(con: duckdb.DuckDBPyConnection) -> bool:
    if not duckdb_table_exists(con, TRANSITION_EVENTS_TABLE) or not duckdb_table_exists(con, TRANSITION_METADATA_TABLE):
        return False
    existing = dict(con.execute(f"SELECT key, value FROM {TRANSITION_METADATA_TABLE}").fetchall())
    return existing == transition_store_signature()


## Load Vocabulary And Artist Scope

A full transition table over all `H_n` classes would be too large and too sparse. This notebook starts with frequent harmonic classes per `n`, which gives a stable first language model and keeps artist-conditioned estimates interpretable.

Artist names are not available in the current raw dataset. The notebook therefore works with anonymized `artist_id` values. The method supports a named artist such as Stevie Wonder as soon as a name-to-`artist_id` mapping is added.

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    vocab = con.execute(
        """
        SELECT *
        FROM (
            SELECT
                n,
                harmonic_id,
                harmonic_key,
                example_ngram,
                count AS global_count,
                frequency AS global_frequency,
                n_exact_ngrams,
                ROW_NUMBER() OVER (PARTITION BY n ORDER BY count DESC, harmonic_id) AS rank_within_n
            FROM harmonic_ngrams
            WHERE n IN (3, 4, 5)
              AND count >= ?
        )
        WHERE rank_within_n <= ?
        ORDER BY n, rank_within_n
        """,
        [MIN_GLOBAL_HARMONIC_COUNT, TOP_HARMONIC_PER_N],
    ).fetchdf()

    artist_scope = con.execute(
        """
        SELECT artist_id, COUNT(*) AS song_count
        FROM song_metadata
        WHERE artist_id IS NOT NULL
        GROUP BY artist_id
        HAVING COUNT(*) >= ?
        ORDER BY song_count DESC, artist_id
        LIMIT ?
        """,
        [MIN_ARTIST_SONGS, MAX_ARTISTS],
    ).fetchdf()
finally:
    con.close()

ARTIST_SCOPE_IDS = set(artist_scope["artist_id"])
VOCAB_BY_N = {int(n): set(sub["harmonic_id"]) for n, sub in vocab.groupby("n")}
VOCAB_LOOKUP = vocab.set_index("harmonic_id").to_dict("index")

vocab.to_csv(OUT_DIR / "conditional_lm_vocabulary.csv", index=False)
artist_scope.to_csv(OUT_DIR / "conditional_lm_artist_scope.csv", index=False)

display(vocab.groupby("n").agg(classes=("harmonic_id", "size"), min_count=("global_count", "min"), max_count=("global_count", "max")))
display(artist_scope.head(10))

## Materialize Or Load Harmonic Transition Events

Earlier notebooks materialize global vocabularies, document-term tables, and interpretation outputs, but they do not yet persist the transition-level object needed for language modeling. This section fills that gap.

On the first run, it streams the raw chord corpus once and writes a reusable DuckDB table:

```text
song_id, split, n, position, source_harmonic_id, target_harmonic_id, decade_1950_plus, main_genre, artist_id
```

On later runs, if the table metadata matches the current notebook parameters, the model reads this table directly instead of recomputing harmonic sequences from raw chord strings. Set `REBUILD_TRANSITION_STORE = True` when changing vocabulary or split parameters and you want to force a rebuild.

In [ ]:
def materialize_transition_store() -> pd.DataFrame:
    signature = transition_store_signature()
    con = duckdb.connect(str(DB_PATH))
    ds.configure_connection(con)
    try:
        if REBUILD_TRANSITION_STORE or not transition_store_is_current(con):
            print(f"Building transition store: {TRANSITION_EVENTS_TABLE}")
            con.execute(f"DROP TABLE IF EXISTS {TRANSITION_EVENTS_TABLE}")
            con.execute(f"DROP TABLE IF EXISTS {TRANSITION_METADATA_TABLE}")
            con.execute(f"DROP TABLE IF EXISTS {TRANSITION_COUNTS_TABLE}")
            con.execute(f"DROP TABLE IF EXISTS {SOURCE_SUPPORT_TABLE}")
            con.execute(f"DROP TABLE IF EXISTS {SCOPE_SUMMARY_TABLE}")
            con.execute(
                f"""
                CREATE TABLE {TRANSITION_EVENTS_TABLE} (
                    song_id BIGINT,
                    split VARCHAR,
                    n INTEGER,
                    position INTEGER,
                    source_harmonic_id VARCHAR,
                    target_harmonic_id VARCHAR,
                    decade_1950_plus VARCHAR,
                    main_genre VARCHAR,
                    artist_id VARCHAR
                )
                """
            )
            rows = []
            processed = 0
            inserted = 0

            def flush_rows() -> None:
                nonlocal rows, inserted
                if not rows:
                    return
                batch = pd.DataFrame(rows)
                con.register("transition_batch", batch)
                con.execute(f"INSERT INTO {TRANSITION_EVENTS_TABLE} SELECT * FROM transition_batch")
                con.unregister("transition_batch")
                inserted += len(batch)
                rows = []

            for chunk in pd.read_csv(RAW_PATH, usecols=["id", "chords", "decade", "main_genre", "artist_id"], chunksize=CHUNKSIZE):
                for row in chunk.itertuples(index=False):
                    if MAX_SONGS is not None and processed >= MAX_SONGS:
                        break
                    split = split_for_song(row.id)
                    tokens = nf.chord_tokens(row.chords)
                    decade_value = str(int(row.decade)) if pd.notna(row.decade) and int(row.decade) >= 1950 else None
                    genre_value = str(row.main_genre) if pd.notna(row.main_genre) else None
                    artist_value = str(row.artist_id) if pd.notna(row.artist_id) and row.artist_id in ARTIST_SCOPE_IDS else None
                    for n in NS:
                        sequence = harmonic_sequence(tokens, n, VOCAB_BY_N[n])
                        for position, (source, target) in enumerate(zip(sequence, sequence[1:])):
                            rows.append(
                                {
                                    "song_id": int(row.id),
                                    "split": split,
                                    "n": int(n),
                                    "position": int(position),
                                    "source_harmonic_id": source,
                                    "target_harmonic_id": target,
                                    "decade_1950_plus": decade_value,
                                    "main_genre": genre_value,
                                    "artist_id": artist_value,
                                }
                            )
                    processed += 1
                    if len(rows) >= 200_000:
                        flush_rows()
                if MAX_SONGS is not None and processed >= MAX_SONGS:
                    break
            flush_rows()
            con.execute(f"CREATE INDEX IF NOT EXISTS {TRANSITION_EVENTS_TABLE}_split_idx ON {TRANSITION_EVENTS_TABLE}(split)")
            con.execute(f"CREATE INDEX IF NOT EXISTS {TRANSITION_EVENTS_TABLE}_n_source_idx ON {TRANSITION_EVENTS_TABLE}(n, source_harmonic_id)")
            con.execute(f"CREATE TABLE {TRANSITION_METADATA_TABLE} (key VARCHAR, value VARCHAR)")
            con.executemany(f"INSERT INTO {TRANSITION_METADATA_TABLE} VALUES (?, ?)", list(signature.items()))
            print(f"Inserted {inserted:,} transition events from {processed:,} songs")
        else:
            print(f"Using existing transition store: {TRANSITION_EVENTS_TABLE}")

        summary = con.execute(
            f"""
            SELECT split, n, COUNT(*) AS transition_events, COUNT(DISTINCT song_id) AS songs
            FROM {TRANSITION_EVENTS_TABLE}
            GROUP BY split, n
            ORDER BY split, n
            """
        ).fetchdf()
    finally:
        con.close()
    return summary

transition_store_summary = materialize_transition_store()
display(transition_store_summary)

## Materialize Scoped Transition Counts

The full scoped count table is large, so keep it in DuckDB rather than pulling millions of rows into pandas. This section creates three reusable aggregate tables:

- `TRANSITION_COUNTS_TABLE`: scoped `source -> target` counts;
- `SOURCE_SUPPORT_TABLE`: total support for each source under each scope;
- `SCOPE_SUMMARY_TABLE`: total event support by split/scope/`n`.

The rest of the notebook reads small summaries or per-source distributions from these tables on demand.

If you interrupted an earlier DuckDB cell, restart the notebook kernel before running this section. DuckDB allows many readers but only one writer, and a stale interrupted kernel can keep the database write lock.

In [ ]:
def materialize_scoped_transition_counts() -> dict[str, int]:
    con = duckdb.connect(str(DB_PATH))
    ds.configure_connection(con)
    try:
        if REBUILD_TRANSITION_STORE or not duckdb_table_exists(con, TRANSITION_COUNTS_TABLE):
            print(f"Building scoped transition counts: {TRANSITION_COUNTS_TABLE}")
            con.execute(f"DROP TABLE IF EXISTS {TRANSITION_COUNTS_TABLE}")
            con.execute(
                f"""
                CREATE TABLE {TRANSITION_COUNTS_TABLE} AS
                SELECT split, 'global' AS scope_type, '__all__' AS scope_value, n, source_harmonic_id, target_harmonic_id, COUNT(*)::BIGINT AS count
                FROM {TRANSITION_EVENTS_TABLE}
                GROUP BY split, n, source_harmonic_id, target_harmonic_id
                UNION ALL
                SELECT split, 'decade_1950_plus' AS scope_type, decade_1950_plus AS scope_value, n, source_harmonic_id, target_harmonic_id, COUNT(*)::BIGINT AS count
                FROM {TRANSITION_EVENTS_TABLE}
                WHERE decade_1950_plus IS NOT NULL
                GROUP BY split, decade_1950_plus, n, source_harmonic_id, target_harmonic_id
                UNION ALL
                SELECT split, 'main_genre' AS scope_type, main_genre AS scope_value, n, source_harmonic_id, target_harmonic_id, COUNT(*)::BIGINT AS count
                FROM {TRANSITION_EVENTS_TABLE}
                WHERE main_genre IS NOT NULL
                GROUP BY split, main_genre, n, source_harmonic_id, target_harmonic_id
                UNION ALL
                SELECT split, 'artist_id' AS scope_type, artist_id AS scope_value, n, source_harmonic_id, target_harmonic_id, COUNT(*)::BIGINT AS count
                FROM {TRANSITION_EVENTS_TABLE}
                WHERE artist_id IS NOT NULL
                GROUP BY split, artist_id, n, source_harmonic_id, target_harmonic_id
                """
            )
            con.execute(f"CREATE INDEX IF NOT EXISTS {TRANSITION_COUNTS_TABLE}_lookup_idx ON {TRANSITION_COUNTS_TABLE}(split, scope_type, scope_value, n, source_harmonic_id)")
            con.execute(f"CREATE INDEX IF NOT EXISTS {TRANSITION_COUNTS_TABLE}_summary_idx ON {TRANSITION_COUNTS_TABLE}(split, scope_type, n)")
        else:
            print(f"Using existing scoped transition counts: {TRANSITION_COUNTS_TABLE}")

        if REBUILD_TRANSITION_STORE or not duckdb_table_exists(con, SOURCE_SUPPORT_TABLE):
            con.execute(f"DROP TABLE IF EXISTS {SOURCE_SUPPORT_TABLE}")
            con.execute(
                f"""
                CREATE TABLE {SOURCE_SUPPORT_TABLE} AS
                SELECT split, scope_type, scope_value, n, source_harmonic_id, SUM(count)::BIGINT AS source_count
                FROM {TRANSITION_COUNTS_TABLE}
                GROUP BY split, scope_type, scope_value, n, source_harmonic_id
                """
            )
            con.execute(f"CREATE INDEX IF NOT EXISTS {SOURCE_SUPPORT_TABLE}_lookup_idx ON {SOURCE_SUPPORT_TABLE}(split, scope_type, scope_value, n, source_harmonic_id)")

        if REBUILD_TRANSITION_STORE or not duckdb_table_exists(con, SCOPE_SUMMARY_TABLE):
            con.execute(f"DROP TABLE IF EXISTS {SCOPE_SUMMARY_TABLE}")
            con.execute(
                f"""
                CREATE TABLE {SCOPE_SUMMARY_TABLE} AS
                SELECT split, scope_type, scope_value, n, SUM(count)::BIGINT AS events
                FROM {TRANSITION_COUNTS_TABLE}
                GROUP BY split, scope_type, scope_value, n
                """
            )

        counts = {
            "transition_count_rows": con.execute(f"SELECT COUNT(*) FROM {TRANSITION_COUNTS_TABLE}").fetchone()[0],
            "source_support_rows": con.execute(f"SELECT COUNT(*) FROM {SOURCE_SUPPORT_TABLE}").fetchone()[0],
            "scope_summary_rows": con.execute(f"SELECT COUNT(*) FROM {SCOPE_SUMMARY_TABLE}").fetchone()[0],
        }
    finally:
        con.close()
    return counts

aggregate_counts = materialize_scoped_transition_counts()

con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    test_events = con.execute(
        f"""
        SELECT
            song_id,
            n,
            source_harmonic_id,
            target_harmonic_id,
            decade_1950_plus AS decade,
            main_genre,
            artist_id
        FROM {TRANSITION_EVENTS_TABLE}
        WHERE split = 'test'
        LIMIT ?
        """,
        [MAX_TEST_EVENTS],
    ).fetchdf().to_dict("records")
finally:
    con.close()

{**aggregate_counts, "test_events_sampled": len(test_events)}

## Inspect Transition Support

Before evaluating predictions, inspect how much evidence exists under each scope. The artist-conditioned model is expected to be sparse; that is why interpolation/backoff is required.

This section answers a practical question: when the model says “artist-conditioned,” is it actually using artist evidence, or is it mostly falling back to broader language?

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    scope_summary = con.execute(f"SELECT * FROM {SCOPE_SUMMARY_TABLE} ORDER BY split, scope_type, n, events DESC").fetchdf()
    top_transitions = con.execute(
        f"""
        SELECT *
        FROM {TRANSITION_COUNTS_TABLE}
        WHERE split = 'train'
        ORDER BY count DESC
        LIMIT 20
        """
    ).fetchdf()
finally:
    con.close()

scope_summary.to_csv(OUT_DIR / "conditional_lm_scope_summary.csv", index=False)
top_transitions["source_example"] = top_transitions["source_harmonic_id"].map(short_label)
top_transitions["target_example"] = top_transitions["target_harmonic_id"].map(short_label)
top_transitions.to_csv(OUT_DIR / "conditional_lm_top_transition_counts.csv", index=False)

display(
    scope_summary
    .groupby(["split", "scope_type", "n"])
    .agg(scopes=("scope_value", "nunique"), events=("events", "sum"), median_events=("events", "median"))
    .reset_index()
)
display(top_transitions[["split", "scope_type", "scope_value", "n", "source_example", "target_example", "count"]])

## Build Conditional Distributions

This is a count-based language model:

$$
p(g' \mid g, s) = \frac{c_s(g \to g')}{\sum_h c_s(g \to h)}.
$$

The interpolated model combines scopes when available:

```text
artist -> genre -> decade -> global
```

Weights are support-aware. A sparse artist source gets downweighted automatically, while a well-supported artist source can dominate the mixture.

In [ ]:
DISTRIBUTION_CACHE: dict[tuple[str, str, int, str], tuple[Counter, int]] = {}
UNIGRAM_CACHE: dict[int, dict[str, float]] = {}
MODEL_CON = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(MODEL_CON)


def component_distribution(scope_type: str, scope_value: str, n: int, source: str) -> tuple[Counter, int]:
    key = (scope_type, scope_value, int(n), source)
    if key not in DISTRIBUTION_CACHE:
        df = MODEL_CON.execute(
            f"""
            SELECT target_harmonic_id, count
            FROM {TRANSITION_COUNTS_TABLE}
            WHERE split = 'train'
              AND scope_type = ?
              AND scope_value = ?
              AND n = ?
              AND source_harmonic_id = ?
            """,
            [scope_type, scope_value, int(n), source],
        ).fetchdf()
        counter = Counter(dict(zip(df["target_harmonic_id"], df["count"].astype(int)))) if not df.empty else Counter()
        DISTRIBUTION_CACHE[key] = (counter, sum(counter.values()))
    return DISTRIBUTION_CACHE[key]


def normalized(counter: Counter) -> dict[str, float]:
    total = sum(counter.values())
    if total <= 0:
        return {}
    return {target: count / total for target, count in counter.items()}


def fallback_unigram(n: int) -> dict[str, float]:
    n = int(n)
    if n not in UNIGRAM_CACHE:
        df = MODEL_CON.execute(
            f"""
            SELECT target_harmonic_id, SUM(count)::BIGINT AS count
            FROM {TRANSITION_COUNTS_TABLE}
            WHERE split = 'train'
              AND scope_type = 'global'
              AND n = ?
            GROUP BY target_harmonic_id
            """,
            [n],
        ).fetchdf()
        UNIGRAM_CACHE[n] = normalized(Counter(dict(zip(df["target_harmonic_id"], df["count"].astype(int)))))
    return UNIGRAM_CACHE[n]


def adaptive_weight(scope_type: str, support: int) -> float:
    prior = INTERPOLATION_PRIORS.get(scope_type, 0.0)
    if scope_type == "global":
        return prior
    return prior * (support / (support + BACKOFF_K))


def predict_distribution(n: int, source: str, *, artist_id=None, main_genre=None, decade=None) -> tuple[dict[str, float], pd.DataFrame]:
    requested = []
    if artist_id:
        requested.append(("artist_id", str(artist_id)))
    if main_genre:
        requested.append(("main_genre", str(main_genre)))
    if decade:
        requested.append(("decade_1950_plus", str(decade)))
    requested.append(("global", "__all__"))

    components = []
    for scope_type, scope_value in requested:
        counter, support = component_distribution(scope_type, scope_value, n, source)
        if support <= 0:
            continue
        weight = adaptive_weight(scope_type, support)
        if weight <= 0:
            continue
        components.append((scope_type, scope_value, support, weight, normalized(counter)))

    if not components:
        probs = fallback_unigram(n)
        component_table = pd.DataFrame([{"scope_type": "unigram_fallback", "scope_value": "__global_targets__", "support": len(probs), "weight": 1.0}])
        return probs, component_table

    weight_total = sum(weight for *_rest, weight, _dist in components)
    probs = Counter()
    component_rows = []
    for scope_type, scope_value, support, raw_weight, dist in components:
        weight = raw_weight / weight_total if weight_total else 0.0
        component_rows.append({"scope_type": scope_type, "scope_value": scope_value, "support": support, "weight": weight})
        for target, p in dist.items():
            probs[target] += weight * p

    total = sum(probs.values())
    if total > 0:
        probs = {target: p / total for target, p in probs.items()}
    return dict(probs), pd.DataFrame(component_rows)


def top_predictions(probabilities: dict[str, float], k: int = 10) -> pd.DataFrame:
    rows = []
    for rank, (target, prob) in enumerate(sorted(probabilities.items(), key=lambda item: (-item[1], item[0]))[:k], start=1):
        rows.append({"rank": rank, "target_harmonic_id": target, "target_example": short_label(target), "probability": prob})
    return pd.DataFrame(rows)

## Evaluate Global Versus Metadata-Conditioned Prediction

Evaluation uses a deterministic song-level holdout. The main metrics are:

- `top1`, `top5`, `top10`: whether the true continuation appears in the top-k predictions;
- `MRR`: reciprocal rank of the true continuation;
- `NLL`: negative log probability assigned to the true continuation.

This is not meant to be the final modeling benchmark. It is a sanity check that conditioning and backoff are behaving sensibly. Because the vocabulary is support-limited, these metrics describe prediction **within the retained harmonic vocabulary**, not every possible chord continuation.

In [ ]:
def score_event(event: dict, model_name: str) -> dict:
    if model_name == "global":
        probs, components = predict_distribution(event["n"], event["source_harmonic_id"])
    elif model_name == "interpolated":
        probs, components = predict_distribution(
            event["n"],
            event["source_harmonic_id"],
            artist_id=event.get("artist_id"),
            main_genre=event.get("main_genre"),
            decade=event.get("decade"),
        )
    else:
        raise ValueError(model_name)

    target = event["target_harmonic_id"]
    sorted_targets = sorted(probs.items(), key=lambda item: (-item[1], item[0]))
    rank = None
    for i, (candidate, _p) in enumerate(sorted_targets, start=1):
        if candidate == target:
            rank = i
            break
    probability = max(probs.get(target, 0.0), 1e-12)
    return {
        "model": model_name,
        "n": event["n"],
        "rank": rank,
        "top1": rank == 1,
        "top5": rank is not None and rank <= 5,
        "top10": rank is not None and rank <= 10,
        "reciprocal_rank": 0.0 if rank is None else 1.0 / rank,
        "nll": -math.log(probability),
        "components": "; ".join(f"{r.scope_type}:{r.scope_value}:{r.weight:.2f}" for r in components.itertuples(index=False)),
    }

scores = []
for event in test_events:
    scores.append(score_event(event, "global"))
    scores.append(score_event(event, "interpolated"))

score_table = pd.DataFrame(scores)
evaluation_summary = (
    score_table
    .groupby(["model", "n"], as_index=False)
    .agg(
        events=("rank", "size"),
        top1=("top1", "mean"),
        top5=("top5", "mean"),
        top10=("top10", "mean"),
        mrr=("reciprocal_rank", "mean"),
        mean_nll=("nll", "mean"),
    )
)

score_table.to_csv(OUT_DIR / "conditional_lm_event_scores.csv", index=False)
evaluation_summary.to_csv(OUT_DIR / "conditional_lm_evaluation_summary.csv", index=False)

display(evaluation_summary)

## Style-Conditioned Continuation

This is the notebook's interactive endpoint. Pick a source harmonic class and metadata, then inspect the predicted next states.

The output includes both predictions and interpolation components. The component table matters: it tells you whether the answer is truly artist-driven or mostly backed off to genre/decade/global evidence.

Because artist names are unavailable in the current dataset, the artist parameter uses anonymized `artist_id`. The Stevie Wonder version of this question becomes possible once a name-to-id mapping is attached.

In [ ]:
def find_context(query: str, n: int | None = None, k: int = 20) -> pd.DataFrame:
    sub = vocab.copy()
    if n is not None:
        sub = sub[sub["n"] == n]
    mask = sub["harmonic_id"].str.contains(query, case=False, na=False) | sub["example_ngram"].str.contains(query, case=False, na=False)
    return sub[mask].head(k)[["n", "harmonic_id", "example_ngram", "global_count", "global_frequency"]]


def suggest_next(source_harmonic_id: str, n: int, *, artist_id=None, main_genre=None, decade=None, k: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    probs, components = predict_distribution(n, source_harmonic_id, artist_id=artist_id, main_genre=main_genre, decade=decade)
    return top_predictions(probs, k=k), components

# Pick a frequent source context and a supported artist for demonstration.
demo_source = vocab[vocab["n"] == 3].iloc[0]["harmonic_id"]
demo_artist = artist_scope.iloc[0]["artist_id"] if not artist_scope.empty else None
demo_genre = "pop"
demo_decade = "2020"

print("Demo source:", short_label(demo_source), demo_source)
print("Demo artist_id:", demo_artist)

pred_global, comp_global = suggest_next(demo_source, 3, k=10)
pred_style, comp_style = suggest_next(demo_source, 3, artist_id=demo_artist, main_genre=demo_genre, decade=demo_decade, k=10)

print("\nGlobal continuation")
display(pred_global)
display(comp_global)

print("\nStyle-conditioned continuation")
display(pred_style)
display(comp_style)

## Compare Continuations Across Styles

The same source context can imply different continuations under different metadata. This section compares the global model, a genre-conditioned model, and an artist-conditioned model side by side.

In [ ]:
comparison_specs = [
    {"label": "global", "kwargs": {}},
    {"label": "genre_pop", "kwargs": {"main_genre": "pop"}},
    {"label": "genre_country", "kwargs": {"main_genre": "country"}},
]
if demo_artist:
    comparison_specs.append({"label": f"artist_{demo_artist}", "kwargs": {"artist_id": demo_artist}})

comparison_rows = []
for spec in comparison_specs:
    pred, components = suggest_next(demo_source, 3, k=10, **spec["kwargs"])
    for row in pred.itertuples(index=False):
        comparison_rows.append({"condition": spec["label"], **row._asdict()})

conditioned_prediction_comparison = pd.DataFrame(comparison_rows)
conditioned_prediction_comparison.to_csv(OUT_DIR / "conditional_lm_style_prediction_examples.csv", index=False)
display(conditioned_prediction_comparison)

### Reading The Comparison

If the global, genre, and artist lists are nearly identical, the source context is probably governed by broad harmonic grammar. If the rankings diverge, the metadata is changing the continuation distribution and the example is worth musical inspection.

## Highest-Confidence Artist Sources

Some artists have enough transition support to make artist-conditioned predictions meaningful. This table finds source contexts where artist-level evidence is strongest.

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    artist_source_support = con.execute(
        f"""
        SELECT *
        FROM {SOURCE_SUPPORT_TABLE}
        WHERE split = 'train'
          AND scope_type = 'artist_id'
          AND source_count >= ?
        ORDER BY source_count DESC
        """,
        [MIN_SCOPE_SOURCE_COUNT],
    ).fetchdf()
finally:
    con.close()

artist_source_support["source_example"] = artist_source_support["source_harmonic_id"].map(short_label)
artist_source_support.to_csv(OUT_DIR / "conditional_lm_supported_artist_sources.csv", index=False)
display(artist_source_support.head(30))

## Export Summary

The main outputs are:

- `conditional_lm_top_transition_counts.csv`: top train transition counts for inspection;
- `conditional_lm_scope_summary.csv`: transition support by scope;
- `conditional_lm_supported_artist_sources.csv`: artist/source support diagnostics;
- `conditional_lm_evaluation_summary.csv`: global vs interpolated prediction metrics;
- `conditional_lm_style_prediction_examples.csv`: example continuations under different metadata;
- `conditional_lm_supported_artist_sources.csv`: artist/source pairs with enough support to inspect.

The next modeling step is to replace pure counts with embedding-aware smoothing from notebook 10: if a source transition is sparse, borrow probability mass from distributionally nearby source contexts.

In [ ]:
outputs = sorted(p.name for p in OUT_DIR.glob("*"))
try:
    MODEL_CON.close()
except NameError:
    pass
outputs